# Validação de Dados - Views BigQuery

Este notebook valida se as views criadas no BigQuery estão retornando os dados corretamente.

> **Dica:** Se receber um erro 404 (Not Found), certifique-se de que rodou o **Notebook 02** primeiro para criar as views!

In [ ]:
from google.cloud import bigquery
import pandas as pd
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Autenticado no Colab!")
except ImportError:
    print("Rodando localmente ou em outro ambiente.")

PROJECT_ID = "directed-mender-489100-m3"
client = bigquery.Client(project=PROJECT_ID)

def run_query(sql):
    return client.query(sql).to_dataframe()

## 1. Verificação de Existência
Vamos listar as tabelas/views no dataset para ver se elas existem.

In [ ]:
dataset_id = f"{PROJECT_ID}.games_analytics"
tables = list(client.list_tables(dataset_id))
print(f"Tabelas encontradas em {dataset_id}:")
for table in tables:
    print(f" - {table.table_id}")

## 2. Comparação de Contagem de Linhas

A `vw_analise_games` (detalhada) deve ter mais linhas que a `vw_analise_games_unica` (agrupada).

In [ ]:
sql_counts = f"""
SELECT 
    'Detalhada' as view, count(*) as total
FROM `{PROJECT_ID}.games_analytics.vw_analise_games` 
UNION ALL
SELECT 
    'Unificada' as view, count(*) as total
FROM `{PROJECT_ID}.games_analytics.vw_analise_games_unica` 
"""
df_counts = run_query(sql_counts)
df_counts

## 3. Teste de Duplicação (Caso Real)
Vamos testar se o agrupamento funcionou para um jogo específico.

In [ ]:
nome_jogo = "The Witcher 3: Wild Hunt"

sql_test = f"""
SELECT nome_jogo, plataformas, generos, distribuidoras 
FROM `{PROJECT_ID}.games_analytics.vw_analise_games_unica` 
WHERE nome_jogo LIKE '%{nome_jogo}%'
LIMIT 1
"""
run_query(sql_test)